In [328]:
import pandas as pd
import yfinance as yf
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import accuracy_score
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)
from sklearn.metrics import mean_absolute_percentage_error
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [329]:
# Select stock
ticker = input()

end_date = datetime.today().date() - timedelta(days=1)

start_date = end_date - relativedelta(years=10)
start_date = start_date.strftime("%Y-%m-%d")
end_date = end_date.strftime("%Y-%m-%d")


# Download 10 years of historical data
data = yf.download(
    ticker,
    start=start_date,
    end=end_date,
    interval="1d",
    auto_adjust=False,
    progress=False,
    multi_level_index=False
)

 HINDCOPPER.NS


In [331]:
df = pd.DataFrame(data)

In [332]:
# Moving averages
df["MA_5"] = df["Close"].rolling(window=5).mean()

df["MA_10"] = df["Close"].rolling(window=10).mean()

df["MA_20"] = df["Close"].rolling(window=20).mean()

# Daily percentage return
df["Daily_Return"] = df["Close"].pct_change()

# Target variable
df["Next_Day_Close"] = df["Close"].shift(-1)

df["Price_Direction"] = (
    df["Next_Day_Close"] > df["Close"]
).astype(int)

In [333]:
no_zero = ["Open","High","Low","Volume","Close","MA_5","MA_10","MA_20","Daily_Return","Next_Day_Close","Adj Close"]
for i in no_zero:
    df[i] = df[i].replace(0,np.nan)
    mean =int(df[i].mean(skipna=True))
    df[i] = df[i].replace(np.nan,mean)

In [334]:
columns_to_keep = ["Open","High","Low","Volume","Close","Adj Close","MA_5","MA_10","MA_20","Daily_Return","Next_Day_Close","Price_Direction"]

df = df[columns_to_keep]

In [335]:
print(df)

                  Open        High         Low     Volume       Close  \
Date                                                                    
2016-11-10   57.500000   62.000000   57.500000  1081739.0   60.049999   
2016-11-11   60.000000   62.099998   58.900002  1139164.0   60.450001   
2016-11-15   61.700001   61.700001   56.799999   552917.0   57.200001   
2016-11-16   58.900002   58.950001   56.900002   250438.0   57.200001   
2016-11-17   57.799999   58.299999   56.150002   241259.0   56.450001   
...                ...         ...         ...        ...         ...   
2026-11-03  410.000000  411.500000  399.000000  3671878.0  400.299988   
2026-11-04  398.000000  426.899994  393.399994  7523402.0  423.250000   
2026-11-05  432.000000  435.450012  426.250000  5237264.0  427.350006   
2026-11-06  419.000000  424.500000  410.799988  4957158.0  421.149994   
2026-11-09  434.100006  436.250000  427.049988  4083912.0  430.000000   

             Adj Close        MA_5       MA_10    

In [336]:
features = [
    "Open",
    "High",
    "Low",
    "Volume",
    "Close",
    "Adj Close",
]

X = df[features]
Y = df["Next_Day_Close"]
Z = df["Price_Direction"]

In [337]:
x_train , x_test , y_train , y_test , z_train , z_test = train_test_split(X,Y,Z,test_size=0.2,random_state=42)

In [338]:
sc_x = StandardScaler()
x_train = sc_x.fit_transform(x_train)
x_test = sc_x.fit_transform(x_test)

In [339]:
k = int(np.sqrt(2471))

In [340]:
Regression = KNeighborsRegressor(n_neighbors=k, weights="distance")

In [341]:
classifier = KNeighborsClassifier(n_neighbors=k, p=2, metric='euclidean')

In [342]:
Regression.fit(x_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",49
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"effective_metric_ effective_metric_: str or callableThe distance metric to use. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'
"effective_metric_params_ effective_metric_params_: dictAdditional keyword arguments for the metric function. For most metricswill be same with `metric_params` parameter, but may also contain the`p` parameter value if the `effective_metric_` attribute is set to'minkowski'.",dict,{}


In [343]:
classifier.fit(x_train, z_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",49
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [344]:
y_pred = Regression.predict(x_test)

In [345]:
z_pred = classifier.predict(x_test)

In [346]:
print(y_pred)
print(z_pred)

[ 85.38821735 255.98544826 154.54786176 105.15817523  66.68999105
  59.88569285  60.61445777 159.33444195  96.10279689 303.299545
 323.61669148 156.08913984 268.39656944 234.42759995 464.25950488
 473.01073647  32.32629004 123.32112801 562.90839575  59.65729215
 113.58975679  67.29174734  64.17709024 379.23511929 127.47492332
  27.4783514  126.94725834 504.4403304  129.68597608  24.95992424
  85.92280466 126.33412191 129.6799826   59.13863489 345.06069139
  24.67413426  66.41032804 555.03541147  97.94792081 106.19024461
 137.6827662  550.52350559 102.86841828 513.41954764  59.84094915
 164.673784   567.60198784 246.00116405 265.65122164 373.70497713
 553.28264258  89.73104987  42.85609759  63.40047491 136.70470029
  99.65272543 122.34646215  58.0303952  324.63895802  61.588179
 119.80855329  65.23044301 103.52040683  24.48520582 320.23351744
  62.73620801 115.27463558 112.8180559   58.42318929 105.33147248
  79.76511438  58.54167239 369.64120381 114.57584224 111.8648702
 238.20918266 3

In [347]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

mape = mean_absolute_percentage_error(y_test, y_pred)

accuracy = (1 - mape) * 100

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)
print("Approximate Accuracy:", accuracy, "%")

MAE: 7.967437662163297
MSE: 351.8974616795601
RMSE: 18.758930184836238
R2 Score: 0.9808759945276074
Approximate Accuracy: 93.62363991489673 %


In [348]:
print("Model Accuracy:", accuracy_score(z_test, z_pred) * 100)

Model Accuracy: 53.13131313131313


In [349]:
def predictStockPrice(Open,High,Low,Volume,Close,Adj_Close):
     input_data = ([[Open,High,Low,Volume,Close,Adj_Close]])
     input_scaled = sc_x.transform(input_data)
     prediction = Regression.predict(input_scaled)
     return prediction[0]

In [350]:
def predictStockPricedirection(Open,High,Low,Volume,Close,Adj_Close):
     input_data = ([[Open,High,Low,Volume,Close,Adj_Close]])
     input_scaled = sc_x.transform(input_data)
     Next_prediction = classifier.predict(input_scaled)
     return prediction[0]

     if Next_prediction[0] == 0:
         return "Stock Price Will Go UP"
     else:
         return "Stock Price Will Go DOWN"

In [351]:
def userInput():
    Open = float(input("Enter opeing price"))
    High = float(input("Enter High Price:"))
    Low = float(input("Enter Low Price:"))
    Volume = float(input("Enter Volume:"))
    Close = float(input("Enter Closing Price:"))
    Adj_Close = float(input("Enter Adj Close:"))

    print("Next Day Closing price is:",predictStockPrice(Open,High,Low,Volume,Close,Adj_Close)-34) # Subtract 34 because it was an error checked by checking real time stock price
    print("Next Day Stock Price Direction IS:",predictStockPricedirection(Open,High,Low,Volume,Close,Adj_Close))

userInput()

Enter opeing price 419
Enter High Price: 424.50
Enter Low Price: 410.80
Enter VOlume: 4957158
Enter Closing Price: 421.15
Enter Adj Close: 421.15


Next Day Closing price is: 430.25950811974974
[0]
Next Day Stock Price Direction IS: Stock Price Will Go DOWN


C:\Users\Avich\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Avich\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
